# Preprocesamiento: etiquetado meteorológico (weather) con AEMET

Etiqueta cada fotograma de la fase `frames` cruzando:
- las coordenadas de su cámara (catálogo `da_camarasTrafico-4326.csv`)
- su marca de tiempo UTC (`timestamp`)

con las observaciones de la estación meteorológica AEMET más cercana.

Flujo:
1. Cargar la fase `frames` y el catálogo de cámaras.
2. Descargar el inventario de estaciones AEMET, convertir sus coordenadas (grados-minutos-segundos, p. ej. `364000N`) a grados decimales y asignar la estación más cercana a cada cámara (distancia Haversine).
3. Descargar observaciones (últimas horas o rango de fechas).
4. Asignar a cada fotograma la observación más cercana en el tiempo (ambas en UTC).
5. Derivar la etiqueta `weather` (sunny, cloudy, rain, fog, windy, unknown) con `src/meteo.py`.

La iluminación (día / crepúsculo / noche) **no** es una clase meteorológica: la calcula `02_preprocess_labelled_basic` por elevación solar (`illumination`).

**Salida:** `data/malaga/fases/weather.parquet` (fase `weather`), unida al resto por `(camera, image)`.

Requisito: API key gratuita de AEMET OpenData (https://opendata.aemet.es/centrodedescargas/altaUsuario) en la variable de entorno `AEMET_API_KEY`.

## 1. Configuración

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import requests

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from catalogo import load_catalog
from config import load_config, path
from meteo import classify_weather, dms_to_decimal, haversine_km
from pipeline import load_phase, save_phase

CFG = load_config()["meteo"]
CACHE_DIR = path("meteo_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

AEMET_API_KEY = os.environ.get("AEMET_API_KEY", "")
MAX_TIME_DIFF_MINUTES = CFG["max_diff_minutos"]
MAX_STATION_DISTANCE_KM = CFG["max_distancia_km"]

print(f"API key configurada: {'si' if AEMET_API_KEY else 'no (solo funcionara con datos en cache)'}")

## 2. Fotogramas y coordenadas de las cámaras

In [ ]:
frames_df = load_phase("frames")
camaras_df = load_catalog()

print(f"Fotogramas: {len(frames_df)} | Cámaras en catálogo: {len(camaras_df)}")
print(f"Cámaras en frames: {frames_df['camera'].nunique()}")
display(camaras_df[["ID", "NOMBRE", "lat", "lon"]].head())

## 3. Inventario de estaciones AEMET y estación más cercana

El inventario da `latitud`/`longitud` en grados-minutos-segundos con hemisferio (`364000N`, `042900W`); `pd.to_numeric` las convertía en NaN y por eso no se asignaba ninguna estación. Se convierten con `meteo.dms_to_decimal`.

In [ ]:
AEMET_BASE = "https://opendata.aemet.es/opendata/api"
HEADERS = {"api_key": AEMET_API_KEY, "Accept": "application/json"}


def aemet_get(endpoint, params=None, cache_name=None):
    """Llama a un endpoint AEMET (dos pasos: obtener URL de datos y descargar datos) con cache en disco."""
    if cache_name:
        cache_path = CACHE_DIR / cache_name
        if cache_path.exists():
            return json.loads(cache_path.read_text(encoding="utf-8"))

    if not AEMET_API_KEY:
        raise RuntimeError("Falta AEMET_API_KEY. Definela como variable de entorno.")

    r = requests.get(f"{AEMET_BASE}{endpoint}", headers=HEADERS, params=params or {}, timeout=60)
    r.raise_for_status()
    meta = r.json()
    if meta.get("estado") != 200 or "datos" not in meta:
        raise RuntimeError(f"Respuesta AEMET inesperada: {meta}")

    r2 = requests.get(meta["datos"], timeout=120)
    r2.raise_for_status()
    data = r2.json()
    if cache_name:
        cache_path.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
    return data


estaciones = aemet_get(
    "/valores/climatologicos/inventarioestaciones/todasestaciones",
    cache_name="inventario_estaciones.json",
)
estaciones_df = pd.DataFrame(estaciones)
estaciones_df["latitud"] = estaciones_df["latitud"].map(dms_to_decimal)
estaciones_df["longitud"] = estaciones_df["longitud"].map(dms_to_decimal)
estaciones_df = estaciones_df.dropna(subset=["latitud", "longitud"])

print(f"Estaciones en inventario: {len(estaciones_df)}")
display(estaciones_df[["indicativo", "nombre", "provincia", "latitud", "longitud"]].head())

In [ ]:
def nearest_station(lat, lon, stations):
    distances = [haversine_km(lat, lon, s_lat, s_lon) for s_lat, s_lon in zip(stations["latitud"], stations["longitud"])]
    best = int(np.argmin(distances))
    return stations.iloc[best], distances[best]


cam_coords = camaras_df.dropna(subset=["lat", "lon"])
nearest = [nearest_station(lat, lon, estaciones_df) for lat, lon in zip(cam_coords["lat"], cam_coords["lon"])]
camera_station = pd.DataFrame({
    "camera": cam_coords["ID"].values,
    "estacion_id": [s["indicativo"] for s, _ in nearest],
    "estacion_nombre": [s["nombre"] for s, _ in nearest],
    "distancia_estacion_km": [d for _, d in nearest],
})

lejanas = camera_station["distancia_estacion_km"] > MAX_STATION_DISTANCE_KM
camera_station.loc[lejanas, "estacion_id"] = None

print(f"Cámaras con estación asignada: {camera_station['estacion_id'].notna().sum()} de {len(camera_station)}")
print(f"Estaciones distintas usadas: {camera_station['estacion_id'].nunique()}")
print(f"Distancia media: {camera_station['distancia_estacion_km'].mean():.1f} km | máxima: {camera_station['distancia_estacion_km'].max():.1f} km")
display(camera_station.sort_values("distancia_estacion_km", ascending=False).head(10))

## 4. Descargar observaciones meteorológicas

Dos modos de descarga:
- **Últimas observaciones** (`/observacion/convencional/todas`): válido si los fotogramas son recientes (últimas ~24 h).
- **Climatológicas diarias** por rango de fechas: para fotogramas históricos (resolución diaria).

Se selecciona automáticamente según la antigüedad de los timestamps del dataset.

In [ ]:
ts_min, ts_max = frames_df["timestamp"].min(), frames_df["timestamp"].max()
now_utc = pd.Timestamp.now(tz="UTC").tz_localize(None)
horas_desde_min = (now_utc - ts_min).total_seconds() / 3600 if pd.notna(ts_min) else np.inf

use_ultimas = horas_desde_min <= 24
print(f"Rango timestamps (UTC): {ts_min} -> {ts_max}")
print(f"Modo descarga: {'ultimas observaciones' if use_ultimas else 'climatologicas diarias (rango)'}")

obs_records = []
if use_ultimas:
    obs = aemet_get("/observacion/convencional/todas", cache_name=f"obs_ultimas_{now_utc:%Y%m%dT%H}.json")
    for o in obs:
        obs_records.append({
            "estacion_id": o.get("idema"),
            "fint": pd.to_datetime(o.get("fint"), errors="coerce", utc=True).tz_localize(None) if o.get("fint") else pd.NaT,
            "prec": o.get("prec"),   # precipitacion acumulada (mm) en la ultima hora
            "vv": o.get("vv"),       # velocidad media del viento (m/s)
            "hr": o.get("hr"),       # humedad relativa (%)
            "ta": o.get("ta"),       # temperatura (C)
            "vis": o.get("vis"),     # visibilidad (m), si esta disponible
            "nub": o.get("nub"),     # nubosidad (octas), si esta disponible
        })
else:
    fecha_ini = ts_min.strftime("%Y-%m-%dT%H:%M:%SUTC")
    fecha_fin = ts_max.strftime("%Y-%m-%dT%H:%M:%SUTC")
    obs = aemet_get(
        f"/valores/climatologicos/diarios/datos/fechaini/{fecha_ini}/fechafin/{fecha_fin}/todasestaciones",
        cache_name=f"obs_diarias_{ts_min.date()}_{ts_max.date()}.json",
    )
    for o in obs:
        obs_records.append({
            "estacion_id": o.get("indicativo"),
            # Dato diario: se sitúa a mediodía UTC para el cruce temporal
            "fint": pd.to_datetime(o.get("fecha"), errors="coerce") + pd.Timedelta(hours=12),
            "prec": o.get("prec"),
            "vv": o.get("velmedia"),
            "hr": o.get("hrMedia"),
            "ta": o.get("tmed"),
            "vis": None,
            "nub": None,
        })

obs_df = pd.DataFrame(obs_records)
for col in ["prec", "vv", "hr", "ta", "vis", "nub"]:
    obs_df[col] = pd.to_numeric(obs_df[col].astype(str).str.replace(",", ".", regex=False), errors="coerce")
obs_df = obs_df.dropna(subset=["estacion_id", "fint"])

# En datos diarios la ventana temporal debe cubrir el día completo
max_diff = MAX_TIME_DIFF_MINUTES if use_ultimas else 12 * 60
print(f"Observaciones descargadas: {len(obs_df)} | ventana de cruce: {max_diff} min")
display(obs_df.head())

## 5. Asignar observación a cada fotograma

In [ ]:
frames_w = frames_df[["camera", "image", "timestamp"]].merge(camera_station, on="camera", how="left")
obs_by_station = {k: g.sort_values("fint") for k, g in obs_df.groupby("estacion_id")}


def nearest_observation(station_id, ts):
    if pd.isna(ts) or station_id not in obs_by_station:
        return None, np.nan
    g = obs_by_station[station_id]
    diffs = (g["fint"] - ts).abs()
    i = diffs.idxmin()
    diff_min = diffs.loc[i].total_seconds() / 60.0
    if diff_min > max_diff:
        return None, diff_min
    return g.loc[i], diff_min


assigned = [nearest_observation(s, ts) for s, ts in zip(frames_w["estacion_id"], frames_w["timestamp"])]
frames_w["obs_asignada"] = [a[0] is not None for a in assigned]
frames_w["diff_obs_minutos"] = [a[1] for a in assigned]
for col in ["prec", "vv", "hr", "ta", "vis", "nub"]:
    frames_w[col] = [a[0][col] if a[0] is not None else np.nan for a in assigned]

n_ok = int(frames_w["obs_asignada"].sum())
print(f"Fotogramas con observación asignada: {n_ok} de {len(frames_w)} ({100 * n_ok / len(frames_w):.1f}%)")
display(frames_w.head(10))

## 6. Derivar la etiqueta weather

Reglas (`meteo.classify_weather`, por orden de prioridad):
1. `rain`: precipitación > 0 mm.
2. `fog`: visibilidad < 1000 m o humedad relativa ≥ 97%.
3. `windy`: viento medio ≥ 10 m/s.
4. `cloudy`: nubosidad ≥ 6 octas o humedad relativa ≥ 85%.
5. `sunny`: resto con observación válida.
6. `unknown`: sin observación asignable.

In [ ]:
frames_w["weather"] = [
    classify_weather(prec=r.prec, vis=r.vis, hr=r.hr, wind=r.vv, nub=r.nub) if r.obs_asignada else "unknown"
    for r in frames_w.itertuples()
]
print("Distribución weather:")
print(frames_w["weather"].value_counts(dropna=False))

## 7. Exportar la fase `weather`

In [ ]:
export_cols = [
    "camera", "image", "weather",
    "estacion_id", "estacion_nombre", "distancia_estacion_km", "diff_obs_minutos",
    "prec", "vv", "hr", "ta", "vis", "nub",
]
out = save_phase(frames_w[export_cols], "weather")
print(f"Fase weather: {len(frames_w)} fotogramas -> {out}")
print(f"Fotogramas con weather desconocido: {(frames_w['weather'] == 'unknown').sum()}")